# EgoCom -> ECMC 正式数据导出

这个 notebook 是正式版预处理流程，用于把 EgoCom 的 transcript 与预提取特征导出为当前 ECMC 项目可用的数据结构。

输出结构：

```text
egocom_ecmc_formal/
  train.csv
  val.csv
  test.csv
  speaker_turns_all.csv
  dataset_report.md
  MMDA/
    split_audio_f/*.npy
    split_video_f/*.npy
```

注意：这里仍然使用占位弱标签。后续需要用 LLM/API 替换 `emotion_bin`、`cognition_bin`、`emotion`、`cognition`。

In [1]:
from pathlib import Path
import csv
import gzip
import json
import math
import pickle
import re
from collections import defaultdict

import numpy as np
import pandas as pd

# =========================
# 路径配置
# =========================
EGO_ROOT = Path(r"F:/egocom")
FEATURE_ROOT = EGO_ROOT / "egocom_pretrained_features.tar" / "egocom_pretrained_features"
TRANSCRIPT_CSV = EGO_ROOT / "ground_truth_transcriptions.csv"
VIDEO_INFO_CSV = EGO_ROOT / "video_info.csv"
FEATURE_CSV_GZ = FEATURE_ROOT / "features_by_history" / "egocom_features_history_4sec.csv.gz"
COLUMN_NAMES_P = FEATURE_ROOT / "feature_column_names.p"

# 输出到新的 formal 目录，不覆盖 debug 目录。
OUT_ROOT = Path(r"D:/py_code/ECMC/my_text/egocom_ecmc_formal")
OUT_MMDA_DIR = OUT_ROOT / "MMDA"
OUT_AUDIO_DIR = OUT_MMDA_DIR / "split_audio_f"
OUT_VIDEO_DIR = OUT_MMDA_DIR / "split_video_f"
OUT_TURNS_CSV = OUT_ROOT / "speaker_turns_all.csv"
OUT_REPORT = OUT_ROOT / "dataset_report.md"

# =========================
# 运行模式
# =========================
# 小样本测试：设置为整数，例如 50。
# 全量导出：设置为 None。
MAX_VIDEO_ROWS_PER_SPLIT = None

# 每个 split 最多导出多少个 turn。小样本测试可设 2000；全量设 None。
MAX_TURNS_PER_SPLIT = None

# speaker turn 切片参数。
MAX_GAP = 1.0
MIN_DURATION = 2.0
MIN_WORDS = 3
MAX_DURATION = 30.0

# coverage 过滤：官方特征在视频尾部可能缺少窗口，低于该比例就跳过。
MIN_FEATURE_COVERAGE = 0.8

# 流式读取大 csv.gz 的 chunk 大小。轻薄本可用 2000-10000。
CHUNKSIZE = 5000

for p in [OUT_ROOT, OUT_AUDIO_DIR, OUT_VIDEO_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("FEATURE_ROOT exists:", FEATURE_ROOT.exists())
print("TRANSCRIPT_CSV exists:", TRANSCRIPT_CSV.exists())
print("VIDEO_INFO_CSV exists:", VIDEO_INFO_CSV.exists())
print("FEATURE_CSV_GZ exists:", FEATURE_CSV_GZ.exists())
print("OUT_ROOT:", OUT_ROOT)

FEATURE_ROOT exists: True
TRANSCRIPT_CSV exists: True
VIDEO_INFO_CSV exists: True
FEATURE_CSV_GZ exists: True
OUT_ROOT: D:\py_code\ECMC\my_text\egocom_ecmc_formal


## 1. 读取特征列

EgoCom 当前特征维度：video 2048 维、audio 512 维、text 300 维。这里先只导出 audio/video，因为当前 ECMC dataloader 的 text 分支直接读取 `content` 文本。

In [2]:
with COLUMN_NAMES_P.open("rb") as f:
    feature_columns = list(pickle.load(f))

video_cols = [c for c in feature_columns if str(c).startswith("videofeat_")]
audio_cols = [c for c in feature_columns if str(c).startswith("voxaudiofeat_")]
text_cols = [c for c in feature_columns if str(c).startswith("textfeat_")]
meta_cols = ["video_id", "clip_id", "video_speaker_id", "is_speaking", "multiclass_speaker_label"]

print("video dim:", len(video_cols))
print("audio dim:", len(audio_cols))
print("text dim:", len(text_cols))

video dim: 2048
audio dim: 512
text dim: 300


## 2. 工具函数

In [3]:
def bool_series(series):
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


def clean_text(words):
    text = " ".join(str(w).strip() for w in words if str(w).strip())
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = text.replace(" ' ", "'")
    text = text.replace(" n' t", "n't")
    return text


def safe_id(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def flush_turn(turn, rows):
    if not turn:
        return
    duration = float(turn["end"] - turn["start"])
    word_count = len(turn["words"])
    content = clean_text(turn["words"])
    if duration < MIN_DURATION or word_count < MIN_WORDS or not content:
        return

    start_ms = int(round(turn["start"] * 1000))
    end_ms = int(round(turn["end"] * 1000))
    conversation_id = safe_id(turn["conversation_id"])
    speaker_id = safe_id(turn["speaker_id"])
    rows.append({
        "id": f"{conversation_id}_spk{speaker_id}_{start_ms}_{end_ms}",
        "conversation_id": turn["conversation_id"],
        "speaker_id": int(turn["speaker_id"]),
        "start": float(turn["start"]),
        "end": float(turn["end"]),
        "duration": duration,
        "word_count": word_count,
        "content": content,
    })


def placeholder_emotion_label(text):
    lower = str(text).lower()
    negative_words = ["hard", "scared", "terrifying", "bad", "hate", "worst", "can't", "cannot"]
    positive_words = ["good", "great", "happy", "awesome", "wonderful", "nice", "love"]
    if any(w in lower for w in negative_words):
        return -1
    if any(w in lower for w in positive_words):
        return 1
    return 0


def split_name_from_row(row):
    if bool(row.get("train", False)):
        return "train"
    if bool(row.get("val", False)):
        return "val"
    if bool(row.get("test", False)):
        return "test"
    return "unknown"

## 3. 构造全部 split 的 speaker turns

In [4]:
def select_video_info_by_split(video_info, split):
    mask = bool_series(video_info[split])
    selected = video_info[mask].copy()
    if MAX_VIDEO_ROWS_PER_SPLIT is not None:
        selected = selected.head(int(MAX_VIDEO_ROWS_PER_SPLIT)).copy()
    return selected


def build_speaker_turns_for_split(transcript_df, video_info, split):
    selected_video_info = select_video_info_by_split(video_info, split)
    wanted_pairs = set(zip(selected_video_info["conversation_id"], selected_video_info["video_speaker_id"].astype(int)))
    if not wanted_pairs:
        return pd.DataFrame()

    df = transcript_df[transcript_df.apply(lambda r: (r["conversation_id"], int(r["speaker_id"])) in wanted_pairs, axis=1)].copy()
    df = df.sort_values(["conversation_id", "startTime", "endTime"]).reset_index(drop=True)

    rows = []
    for conversation_id, group in df.groupby("conversation_id", sort=False):
        current = None
        for row in group.itertuples(index=False):
            speaker_id = int(row.speaker_id)
            start = float(row.startTime)
            end = float(row.endTime)
            word = str(row.word).strip()
            if not word:
                continue

            if current is None:
                current = {"conversation_id": conversation_id, "speaker_id": speaker_id, "start": start, "end": end, "words": [word]}
                continue

            gap = start - current["end"]
            duration_if_added = end - current["start"]
            should_split = speaker_id != current["speaker_id"] or gap > MAX_GAP or duration_if_added > MAX_DURATION
            if should_split:
                flush_turn(current, rows)
                current = {"conversation_id": conversation_id, "speaker_id": speaker_id, "start": start, "end": end, "words": [word]}
            else:
                current["end"] = max(current["end"], end)
                current["words"].append(word)
        flush_turn(current, rows)

    turns = pd.DataFrame(rows)
    if turns.empty:
        return turns

    turns = turns.merge(
        selected_video_info[["video_id", "conversation_id", "video_speaker_id", "train", "val", "test"]],
        left_on=["conversation_id", "speaker_id"],
        right_on=["conversation_id", "video_speaker_id"],
        how="left",
    )
    turns = turns.dropna(subset=["video_id"]).copy()
    turns["video_id"] = turns["video_id"].astype(int)
    turns["split"] = split
    turns["start_clip"] = np.floor(turns["start"]).astype(int)
    turns["end_clip"] = np.ceil(turns["end"]).astype(int)
    if MAX_TURNS_PER_SPLIT is not None:
        turns = turns.head(int(MAX_TURNS_PER_SPLIT)).copy()
    return turns.reset_index(drop=True)


video_info = pd.read_csv(VIDEO_INFO_CSV)
transcript_df = pd.read_csv(TRANSCRIPT_CSV)
transcript_df = transcript_df.dropna(subset=["conversation_id", "speaker_id", "startTime", "endTime", "word"])
transcript_df["speaker_id"] = pd.to_numeric(transcript_df["speaker_id"], errors="coerce")
transcript_df["startTime"] = pd.to_numeric(transcript_df["startTime"], errors="coerce")
transcript_df["endTime"] = pd.to_numeric(transcript_df["endTime"], errors="coerce")
transcript_df = transcript_df.dropna(subset=["speaker_id", "startTime", "endTime"])
transcript_df["speaker_id"] = transcript_df["speaker_id"].astype(int)

turn_parts = []
for split in ["train", "val", "test"]:
    part = build_speaker_turns_for_split(transcript_df, video_info, split)
    print(split, "turns:", len(part), "videos:", part["video_id"].nunique() if len(part) else 0)
    turn_parts.append(part)

turns_all = pd.concat(turn_parts, ignore_index=True)
turns_all.to_csv(OUT_TURNS_CSV, index=False, encoding="utf-8-sig")
print("all turns:", len(turns_all))
print("saved:", OUT_TURNS_CSV)
turns_all.head()

train turns: 3961 videos: 130
val turns: 318 videos: 17
test turns: 787 videos: 26
all turns: 5066
saved: D:\py_code\ECMC\my_text\egocom_ecmc_formal\speaker_turns_all.csv


,id,conversation_id,speaker_id,start,end,duration,word_count,content,video_id,video_speaker_id,train,val,test,split,start_clip,end_clip
0,day_1__con_1__part1_spk1_90_7640,day_1__con_1__part1,1,0.09,7.64,7.55,39,"Okay. So, I have some topics in my hand and I'...",1,1,True,False,False,train,0,8
1,day_1__con_1__part1_spk3_20370_22610,day_1__con_1__part1,3,20.37,22.61,2.24,19,"Curtis, why didn't you wear pants today? Then ...",11,3,True,False,False,train,20,23
2,day_1__con_1__part1_spk2_25920_28010,day_1__con_1__part1,2,25.92,28.01,2.09,11,"The office is always so cold, though. Like -",6,2,True,False,False,train,25,29
3,day_1__con_1__part1_spk2_28220_32770,day_1__con_1__part1,2,28.22,32.77,4.55,36,"... I go outside and I'm like, "" I wish I were...",6,2,True,False,False,train,28,33
4,day_1__con_1__part1_spk1_38060_40900,day_1__con_1__part1,1,38.06,40.90,2.84,16,"Um, what else? What is a - what's a second thing?",1,1,True,False,False,train,38,41


## 4. 构建目标 clip 索引

In [5]:
def build_target_maps(turns_df):
    target_pairs = set()
    turn_to_pairs = {}
    pair_to_turns = defaultdict(list)
    for row in turns_df.itertuples(index=False):
        pairs = []
        for clip_id in range(int(row.start_clip), int(row.end_clip) + 1):
            pair = (int(row.video_id), int(clip_id))
            pairs.append(pair)
            target_pairs.add(pair)
            pair_to_turns[pair].append(row.id)
        turn_to_pairs[row.id] = pairs
    return target_pairs, turn_to_pairs, pair_to_turns


target_pairs, turn_to_pairs, pair_to_turns = build_target_maps(turns_all)
print("target clip pairs:", len(target_pairs))
print("turns:", len(turn_to_pairs))

target clip pairs: 34411
turns: 5066


## 5. 流式读取 EgoCom 特征大表

这一步会扫描 `egocom_features_history_4sec.csv.gz`。全量时可能需要一些时间，但不会把整个 CSV 解压到硬盘。

In [6]:
def load_needed_feature_rows(feature_csv_gz, target_pairs):
    usecols = meta_cols + video_cols + audio_cols
    needed = {}
    total_seen = 0
    total_kept = 0

    for chunk_idx, chunk in enumerate(pd.read_csv(feature_csv_gz, usecols=usecols, chunksize=CHUNKSIZE)):
        chunk["video_id"] = chunk["video_id"].astype(int)
        chunk["clip_id"] = chunk["clip_id"].astype(int)
        mask = [(int(v), int(c)) in target_pairs for v, c in zip(chunk["video_id"], chunk["clip_id"])]
        kept = chunk.loc[mask]
        for row in kept.itertuples(index=False):
            key = (int(row.video_id), int(row.clip_id))
            video = np.array([getattr(row, c) for c in video_cols], dtype=np.float32)
            audio = np.array([getattr(row, c) for c in audio_cols], dtype=np.float32)
            needed[key] = {"video": video, "audio": audio}
        total_seen += len(chunk)
        total_kept += len(kept)
        if chunk_idx % 50 == 0:
            print(f"chunk={chunk_idx}, seen={total_seen}, kept={total_kept}, loaded={len(needed)}/{len(target_pairs)}")
        if len(needed) >= len(target_pairs):
            break
    print(f"finished: seen={total_seen}, kept={total_kept}, loaded={len(needed)}/{len(target_pairs)}")
    return needed


feature_rows = load_needed_feature_rows(FEATURE_CSV_GZ, target_pairs)

chunk=0, seen=5000, kept=1196, loaded=1196/34411
finished: seen=136431, kept=34067, loaded=34067/34411


## 6. 导出 train/val/test CSV 与 npy 特征

In [7]:
def export_dataset(turns_df, feature_rows, turn_to_pairs):
    split_rows = {"train": [], "val": [], "test": []}
    skipped = []

    for row in turns_df.itertuples(index=False):
        pairs = turn_to_pairs[row.id]
        audio_seq = []
        video_seq = []
        for pair in pairs:
            item = feature_rows.get(pair)
            if item is None:
                continue
            audio_seq.append(item["audio"])
            video_seq.append(item["video"])

        expected_len = len(pairs)
        coverage = len(audio_seq) / max(expected_len, 1)
        if not audio_seq or not video_seq or coverage < MIN_FEATURE_COVERAGE:
            skipped.append({"id": row.id, "reason": "low_feature_coverage", "coverage": coverage, "expected_len": expected_len, "actual_len": len(audio_seq)})
            continue

        audio_arr = np.stack(audio_seq).astype(np.float32)
        video_arr = np.stack(video_seq).astype(np.float32)
        np.save(OUT_AUDIO_DIR / f"{row.id}.npy", audio_arr)
        np.save(OUT_VIDEO_DIR / f"{row.id}.npy", video_arr)

        out_row = {
            "id": row.id,
            "content": row.content,
            "emotion_bin": placeholder_emotion_label(row.content),
            "cognition_bin": "[0, 0, 0, 0]",
            "emotion": "占位情绪描述：该片段的真实 emotion caption 需要后续由 LLM/人工校验生成。",
            "cognition": "占位认知描述：该片段的真实 cognition-style caption 需要后续由 LLM/人工校验生成。",
            "conversation_id": row.conversation_id,
            "speaker_id": int(row.speaker_id),
            "video_id": int(row.video_id),
            "start": float(row.start),
            "end": float(row.end),
            "feature_len": len(audio_seq),
            "expected_feature_len": expected_len,
            "feature_coverage": round(coverage, 4),
        }
        split_rows[row.split].append(out_row)

    out_dfs = {}
    for split, rows in split_rows.items():
        df = pd.DataFrame(rows)
        df.to_csv(OUT_ROOT / f"{split}.csv", index=False, encoding="utf-8-sig")
        out_dfs[split] = df
        print(split, "rows:", len(df), "saved:", OUT_ROOT / f"{split}.csv")

    skipped_df = pd.DataFrame(skipped)
    skipped_df.to_csv(OUT_ROOT / "skipped_turns.csv", index=False, encoding="utf-8-sig")
    print("skipped:", len(skipped_df))
    return out_dfs, skipped_df


out_dfs, skipped_df = export_dataset(turns_all, feature_rows, turn_to_pairs)

train rows: 3909 saved: D:\py_code\ECMC\my_text\egocom_ecmc_formal\train.csv
val rows: 312 saved: D:\py_code\ECMC\my_text\egocom_ecmc_formal\val.csv
test rows: 760 saved: D:\py_code\ECMC\my_text\egocom_ecmc_formal\test.csv
skipped: 85


## 7. 质量检查与报告

In [8]:
def inspect_outputs(out_dfs):
    audio_files = list(OUT_AUDIO_DIR.glob("*.npy"))
    video_files = list(OUT_VIDEO_DIR.glob("*.npy"))
    rows = []
    missing = []
    bad = []
    nan_audio = 0
    nan_video = 0
    t_values = []

    all_df = pd.concat(out_dfs.values(), ignore_index=True) if out_dfs else pd.DataFrame()
    for row in all_df.itertuples(index=False):
        ap = OUT_AUDIO_DIR / f"{row.id}.npy"
        vp = OUT_VIDEO_DIR / f"{row.id}.npy"
        if not ap.exists() or not vp.exists():
            missing.append(row.id)
            continue
        a = np.load(ap)
        v = np.load(vp)
        if a.ndim != 2 or v.ndim != 2 or a.shape[1] != len(audio_cols) or v.shape[1] != len(video_cols) or a.shape[0] != v.shape[0]:
            bad.append((row.id, a.shape, v.shape))
        nan_audio += int(np.isnan(a).sum())
        nan_video += int(np.isnan(v).sum())
        t_values.append(a.shape[0])

    summary = {
        "train_rows": len(out_dfs.get("train", [])),
        "val_rows": len(out_dfs.get("val", [])),
        "test_rows": len(out_dfs.get("test", [])),
        "total_rows": len(all_df),
        "audio_files": len(audio_files),
        "video_files": len(video_files),
        "missing_files": len(missing),
        "bad_shapes": len(bad),
        "nan_audio": nan_audio,
        "nan_video": nan_video,
        "audio_dim": len(audio_cols),
        "video_dim": len(video_cols),
        "t_min": int(np.min(t_values)) if t_values else 0,
        "t_mean": float(np.mean(t_values)) if t_values else 0,
        "t_max": int(np.max(t_values)) if t_values else 0,
    }
    return summary, missing, bad


summary, missing, bad = inspect_outputs(out_dfs)
summary

{'train_rows': 3909,
 'val_rows': 312,
 'test_rows': 760,
 'total_rows': 4981,
 'audio_files': 4981,
 'video_files': 4981,
 'missing_files': 0,
 'bad_shapes': 0,
 'nan_audio': 0,
 'nan_video': 0,
 'audio_dim': 512,
 'video_dim': 2048,
 't_min': 4,
 't_mean': 7.031921300943585,
 't_max': 32}

In [9]:
def write_report(out_dfs, skipped_df, summary):
    lines = []
    lines.append("# EgoCom ECMC 数据导出报告")
    lines.append("")
    lines.append("## 配置")
    lines.append("")
    lines.append(f"- MAX_VIDEO_ROWS_PER_SPLIT: `{MAX_VIDEO_ROWS_PER_SPLIT}`")
    lines.append(f"- MAX_TURNS_PER_SPLIT: `{MAX_TURNS_PER_SPLIT}`")
    lines.append(f"- MIN_FEATURE_COVERAGE: `{MIN_FEATURE_COVERAGE}`")
    lines.append(f"- 特征文件: `{FEATURE_CSV_GZ}`")
    lines.append("")
    lines.append("## 样本数量")
    lines.append("")
    for split in ["train", "val", "test"]:
        df = out_dfs.get(split, pd.DataFrame())
        lines.append(f"- {split}: `{len(df)}`")
    lines.append(f"- skipped: `{len(skipped_df)}`")
    lines.append("")
    lines.append("## 特征检查")
    lines.append("")
    for k, v in summary.items():
        lines.append(f"- {k}: `{v}`")
    lines.append("")
    lines.append("## 说明")
    lines.append("")
    lines.append("当前 `emotion/cognition` 字段仍是占位弱标签。后续需要用 LLM/API 或人工校验替换。")
    lines.append("EgoCom 特征维度为 audio 512、video 2048；训练 ECMC 前需要在模型中加入 adapter 到 768 维。")
    OUT_REPORT.write_text("\n".join(lines), encoding="utf-8")
    print("saved report:", OUT_REPORT)


write_report(out_dfs, skipped_df, summary)
print(OUT_REPORT.read_text(encoding="utf-8"))

saved report: D:\py_code\ECMC\my_text\egocom_ecmc_formal\dataset_report.md
# EgoCom ECMC 数据导出报告

## 配置

- MAX_VIDEO_ROWS_PER_SPLIT: `None`
- MAX_TURNS_PER_SPLIT: `None`
- MIN_FEATURE_COVERAGE: `0.8`
- 特征文件: `F:\egocom\egocom_pretrained_features.tar\egocom_pretrained_features\features_by_history\egocom_features_history_4sec.csv.gz`

## 样本数量

- train: `3909`
- val: `312`
- test: `760`
- skipped: `85`

## 特征检查

- train_rows: `3909`
- val_rows: `312`
- test_rows: `760`
- total_rows: `4981`
- audio_files: `4981`
- video_files: `4981`
- missing_files: `0`
- bad_shapes: `0`
- nan_audio: `0`
- nan_video: `0`
- audio_dim: `512`
- video_dim: `2048`
- t_min: `4`
- t_mean: `7.031921300943585`
- t_max: `32`

## 说明

当前 `emotion/cognition` 字段仍是占位弱标签。后续需要用 LLM/API 或人工校验替换。
EgoCom 特征维度为 audio 512、video 2048；训练 ECMC 前需要在模型中加入 adapter 到 768 维。


## 8. 后续：LLM 弱标注字段

正式训练前，需要把占位字段替换为弱标注：

- `emotion_bin`: `-1 / 0 / 1`，分别表示 negative / neutral / positive。
- `cognition_bin`: 四维列表字符串，如 `[0, 1, 0, 0]`。
- `emotion`: 情绪描述。
- `cognition`: 认知风格描述。

因为当前使用的是预提取特征，弱标注主要来自 transcript，不能声称使用了 facial/vocal cues。